# 2D Implicit Splines — Loading Polygon Data from File

Demonstrates how to load polygon vertices from a plain-text file and
visualise the implicit spline, including contour and surface plots.

**Reference:**  
Li, Q. & Tian, J. (2009). *2D Piecewise Algebraic Splines for Implicit Modeling.*  
ACM Transactions on Graphics, 28(3). DOI: [10.1145/1516522.1516524](https://doi.org/10.1145/1516522.1516524)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QL-UoHull/Implicit-Spline/blob/main/notebooks/02_data_from_file.ipynb)


## 1  Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Implicit-Spline'):
        os.system('git clone https://github.com/QL-UoHull/Implicit-Spline.git')
    sys.path.insert(0, 'Implicit-Spline/python')
else:
    python_dir = os.path.join('..', 'python')
    sys.path.insert(0, python_dir)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from implicit_spline import imp_spline_2d
from implicit_spline.visualization import draw_imp_spline, draw_surface, make_grid

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})
print('Imports OK')


## 2  Load polygon vertices from a text file

The file `data/sample_polygon.txt` ships with the repository.  Each line
contains one vertex: two space-separated floats `x y`.
Lines beginning with `#` are comments.

In [ ]:
# Resolve path relative to repository root (works locally and in Colab)
if IN_COLAB:
    data_file = 'Implicit-Spline/data/sample_polygon.txt'
else:
    data_file = os.path.join('..', 'data', 'sample_polygon.txt')

P = np.loadtxt(data_file, comments='#')
print(f'Loaded {len(P)} vertices:')
print(P)


In [ ]:
# Preview the polygon
P_closed = np.vstack([P, P[0]])
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(P_closed[:, 0], P_closed[:, 1], 'b-o', linewidth=2, markersize=8)
for i, (xi, yi) in enumerate(P):
    ax.annotate(f' v{i}', (xi, yi), fontsize=9)
ax.set_aspect('equal'); ax.grid(True)
ax.set_title('Loaded polygon'); ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()


## 3  Set spline parameters

| Parameter | Meaning |
|-----------|--------|
| `delta`   | Transition bandwidth — controls how quickly the function falls from 1 to 0 near the boundary. Typically 5–20 % of polygon size. |
| `n`       | Smoothness order — the result is C^n near each edge. Higher → smoother, but wider transition for same delta. |
| `N`       | Grid resolution for visualisation. |

In [ ]:
delta = 0.15   # try: 0.05, 0.15, 0.30
n     = 2      # try: 1, 2, 3
N     = 300    # grid resolution


## 4  Contour plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
Z, X, Y, ax = draw_imp_spline(P, delta=delta, n=n, N=N, ax=ax,
                               title=f'Sample polygon  (δ={delta}, n={n})')
plt.tight_layout(); plt.show()


## 5  Surface plot

In [ ]:
fig = plt.figure(figsize=(7, 5))
ax3d = fig.add_subplot(111, projection='3d')
draw_surface(P, delta=delta, n=n, N=100, ax=ax3d)
plt.tight_layout(); plt.show()


## 6  Your own polygon

You can either:
* Edit `data/sample_polygon.txt` and re-run cells above, or
* Supply vertices programmatically as shown below.

In [ ]:
# Example: star-like polygon (convex hull of two rotated squares)
theta_outer = np.linspace(0, 2*np.pi, 9)[:-1]        # 8 outer points
theta_inner = theta_outer + np.pi / 8                 # 8 inner points
r_outer, r_inner = 2.0, 1.2
xs = np.concatenate([r_outer*np.cos(theta_outer), r_inner*np.cos(theta_inner)])
ys = np.concatenate([r_outer*np.sin(theta_outer), r_inner*np.sin(theta_inner)])
order = np.argsort(np.arctan2(ys, xs))
P_star = np.column_stack([xs[order], ys[order]])

fig, ax = plt.subplots(figsize=(6, 5))
draw_imp_spline(P_star, delta=0.15, n=2, N=300, ax=ax,
                title='16-vertex star polygon')
plt.tight_layout(); plt.show()


## 7  Direct grid evaluation and custom plotting

For full control, call `imp_spline_2d` directly on a NumPy meshgrid.

In [ ]:
X, Y = make_grid(P, N=300, pad_fraction=0.25)
Z    = imp_spline_2d(X, Y, P, delta=0.12, n=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: filled contour with several iso-levels highlighted
ax = axes[0]
cf = ax.contourf(X, Y, Z, levels=20, cmap='RdYlGn')
plt.colorbar(cf, ax=ax)
for level in [0.1, 0.3, 0.5, 0.7, 0.9]:
    cs = ax.contour(X, Y, Z, levels=[level], colors='k', linewidths=0.8)
    ax.clabel(cs, fmt=f'{level:.1f}', fontsize=8)
ax.set_aspect('equal'); ax.set_title('Multiple iso-levels')
ax.set_xlabel('x'); ax.set_ylabel('y')

# Right: gradient magnitude (norm of numerical gradient)
ax = axes[1]
dZdy, dZdx = np.gradient(Z, Y[:, 0], X[0, :])
grad_mag = np.sqrt(dZdx**2 + dZdy**2)
cf2 = ax.contourf(X, Y, grad_mag, levels=20, cmap='hot_r')
plt.colorbar(cf2, ax=ax)
ax.contour(X, Y, Z, levels=[0.5], colors='cyan', linewidths=1.5)
ax.set_aspect('equal'); ax.set_title('Gradient magnitude  |∇f|')
ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout(); plt.show()
print(f'Z range: [{Z.min():.4f}, {Z.max():.4f}]')
